In [1]:
%load_ext autoreload
%autoreload 2

import torch
from pathlib import Path

In [2]:
from tqdm.notebook import tqdm_notebook as tqdm
import time

total_epochs = 100
epoch_bar = tqdm(range(total_epochs), desc="Epochs")

for i in epoch_bar:
    time.sleep(0.01)

Epochs:   0%|          | 0/100 [00:00<?, ?it/s]

### Preprocess Data

In [3]:
from preprocess_data import preprocess_data
from pathlib import Path

DATASET = "large"

if DATASET == "large":
    src_folder = Path("Results_large/")
    target_folder = Path("dataset/beam_large/")
else:
    raise ValueError(f"Unknown DATASET: {DATASET}")
print("Preprocessing training data...")
preprocess_data(
    data_dir=src_folder,
    split="train",
    noise_scale=0.0003,
    recalc_velocities=True,
    target_dir=target_folder,
)

print()
print("Preprocessing test data...")
preprocess_data(
    data_dir=src_folder, split="val", recalc_velocities=True, target_dir=target_folder
)

Preprocessing training data...
Processing file: Results_large/train/graphs/graphsL0.5_W0.06_D0.1_NL8_NW4_ND4_E1000.0_nu0.3_rho1.0_em0.01_ek0.01_Pix0.0_Piy0.25_Piz0.0_T4.0_Tc0.8_Nsteps50.pt (1/216)
Processing file: Results_large/train/graphs/graphsL0.5_W0.06_D0.1_NL8_NW4_ND4_E1000.0_nu0.3_rho1.0_em0.01_ek0.01_Pix0.0_Piy0.5_Piz0.0_T4.0_Tc0.8_Nsteps50.pt (2/216)
Processing file: Results_large/train/graphs/graphsL0.5_W0.06_D0.1_NL8_NW4_ND4_E1000.0_nu0.3_rho1.0_em0.01_ek0.01_Pix0.0_Piy0.75_Piz0.0_T4.0_Tc0.8_Nsteps50.pt (3/216)
Processing file: Results_large/train/graphs/graphsL0.5_W0.06_D0.1_NL8_NW4_ND4_E1000.0_nu0.3_rho1.0_em0.01_ek0.01_Pix0.0_Piy1.0_Piz0.0_T4.0_Tc0.8_Nsteps50.pt (4/216)
Processing file: Results_large/train/graphs/graphsL0.5_W0.06_D0.1_NL8_NW4_ND4_E1000.0_nu0.3_rho1.0_em0.01_ek0.01_Pix0.0_Piy1.25_Piz0.0_T4.0_Tc0.8_Nsteps50.pt (5/216)
Processing file: Results_large/train/graphs/graphsL0.5_W0.06_D0.1_NL8_NW4_ND4_E1000.0_nu0.3_rho1.0_em0.01_ek0.01_Pix0.0_Piy1.5_Piz0.0_T4.0_Tc

### Load Dataset

In [4]:
from in_memory_dataset import InMemoryTimeStepDataset
from torch_geometric.loader import DataLoader

if DATASET == "large":
    target_folder = Path("dataset/beam_large/")
else:
    raise ValueError(f"Unknown DATASET: {DATASET}")
train_dataset = InMemoryTimeStepDataset(sample_dir=target_folder / "train")
test_dataset = InMemoryTimeStepDataset(sample_dir=target_folder / "val")

batch_size = 16
num_workers = 4
train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

Found 216 sample files
Loaded the dataset with 10584 samples
Found 1 sample files
Loaded the dataset with 49 samples


### Initialize Model

In [5]:
from models.vinay_mgn import MeshGraphNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

node_channels = 13
edge_channels = 8
num_messages = 8
latent_dim = 128

torch.manual_seed(42)
model = MeshGraphNet(
    node_channels=node_channels,
    edge_channels=edge_channels,
    latent_size=latent_dim,
    num_msgs=num_messages,
)
_ = model.to(device)

### Initialize Trainer

In [6]:
from trainer import Trainer

lr = 5e-5
loss_type = "mse"

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
trainer = Trainer(model, optimizer, device, loss_type=loss_type, use_wandb=True)
torch.cuda.empty_cache()
print(f"Training_id: {trainer.training_id}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kevinsteiner to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Training_id: 2026-01-27_14-20-23


### Train Loop

In [7]:
from tqdm.notebook import tqdm_notebook as tqdm
from preprocess_data import Stats
import json

print("Loading stats...")
with open("Results/train/stats/stats.json", "r") as f:
    stats = json.load(f)
node_stats = Stats.from_dict(stats["node"])
edge_stats = Stats.from_dict(stats["edge"])
target_stats = Stats.from_dict(stats["target"])
dt = 0.08

total_epochs = 300
validation_interval = 20

epoch_bar = tqdm(range(total_epochs), desc="Epochs")
for epoch in epoch_bar:
    for batch in tqdm(train_dataloader, desc="Batches", leave=False):
        # print("hey")
        batch.to(device)
        trainer.train(
            batch,
            node_stats=node_stats,
            edge_stats=edge_stats,
            target_stats=target_stats,
        )

    if epoch % validation_interval == 0:
        trainer.test(
            test_loader=test_dataloader,
            node_stats=node_stats,
            edge_stats=edge_stats,
            target_stats=target_stats,
            dt=dt,
            epoch=epoch,
        )
        trainer.save_model(epoch=epoch)
    trainer.epoch_end(epoch=epoch)
    epoch_bar.set_postfix(
        {
            "last_loss": f"{trainer.loss:.6f}, full_rollout_error: {trainer.rollout_all_step_error:.6f}",
        }
    )


Loading stats...


Epochs:   0%|          | 0/300 [00:00<?, ?it/s]

Batches:   0%|          | 0/662 [00:00<?, ?it/s]

/scratch/imos-students/ksteiner/BeamElastoDynamics/trainer.py:96: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  Volume = torch.tensor(graph.volume).squeeze()
/scratch/imos-students/ksteiner/BeamElastoDynamics/trainer.py:97: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:278.)
  cells = torch.tensor(graph.cells).squeeze().to(torch.long)


Rendering frame 0/48
Rendering frame 1/48
Rendering frame 2/48
Rendering frame 3/48
Rendering frame 4/48
Rendering frame 5/48
Rendering frame 6/48
Rendering frame 7/48
Rendering frame 8/48
Rendering frame 9/48
Rendering frame 10/48
Rendering frame 11/48
Rendering frame 12/48
Rendering frame 13/48
Rendering frame 14/48
Rendering frame 15/48
Rendering frame 16/48
Rendering frame 17/48
Rendering frame 18/48
Rendering frame 19/48
Rendering frame 20/48
Rendering frame 21/48
Rendering frame 22/48
Rendering frame 23/48
Rendering frame 24/48
Rendering frame 25/48
Rendering frame 26/48
Rendering frame 27/48
Rendering frame 28/48
Rendering frame 29/48
Rendering frame 30/48
Rendering frame 31/48
Rendering frame 32/48
Rendering frame 33/48
Rendering frame 34/48
Rendering frame 35/48
Rendering frame 36/48
Rendering frame 37/48
Rendering frame 38/48
Rendering frame 39/48
Rendering frame 40/48
Rendering frame 41/48
Rendering frame 42/48
Rendering frame 43/48
Rendering frame 44/48
Rendering frame 45/4

...

Batches:   0%|          | 0/662 [00:00<?, ?it/s]

KeyboardInterrupt: 

socket.send() raised exception.


Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7b396f042b90>> (for post_run_cell), with arguments args (<ExecutionResult object at 7b396f040c70, execution_count=7 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7b396f041c60, raw_cell="from tqdm.notebook import tqdm_notebook as tqdm
fr.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://k8s-container%2B7b22636f6e74657874223a227263702d636161732d70726f64222c22706f646e616d65223a22696e7465726163746976652d746573742d302d30222c226e616d657370616365223a2272756e61692d696d6f732d6b737465696e6572222c226e616d65223a22696e7465726163746976652d74657374222c22696d616765223a2272656769737472792e7263702e6570666c2e63682f696d6f732d6b737465696e65722f626173655f696d6167653a6c6174657374227d/scratch/imos-students/ksteiner/BeamElastoDynamics/example_train.ipynb#X14sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

### Show Training Progress

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

total_epochs = 440
len(train_dataloader)
epochs_indices = np.arange(
    0, total_epochs * len(train_dataloader), len(train_dataloader)
)
# print(epochs_indices)
plt.figure(figsize=(16, 6))
# plt.plot(trainer.train_history, label="Train loss", alpha=0.1, color="gray")
# plt.plot(epochs_indices, trainer.gen_test_history, label="Gen Test Loss")
acc_loss = np.array(trainer.train_acc_history).mean(axis=1)
stress_loss = np.array(trainer.train_stress_history).mean(axis=1)
plt.plot(epochs_indices, acc_loss, label="Acceleration Loss")
plt.plot(epochs_indices, stress_loss, label="Stress Loss")
total_loss = acc_loss + stress_loss
plt.plot(epochs_indices, total_loss, label="Total Loss", linestyle="--", color="black")
plt.legend()
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.xticks(epochs_indices, [str(i) for i in range(total_epochs)])
plt.grid()
plt.show()

NameError: name 'train_dataloader' is not defined

### Show animation

In [ ]:
from rollout_utils import do_rollout

model = trainer.model.eval()
no_rollout = False
true_rollout, pred_rollout = do_rollout(
    model=model,
    test_loader=test_dataloader,
    device=trainer.device,
    node_stats=node_stats,
    edge_stats=edge_stats,
    target_stats=target_stats,
    dt=dt,
    skip_first=0,
    rollout_steps=30,
    dont_rollout=no_rollout,
)

In [ ]:
from make_gif import make_beam_comparison_gif


make_beam_comparison_gif(
    pred_rollout=pred_rollout,
    true_rollout=true_rollout,
    L=1.0,
    W=0.1,
    D=0.04,  # Beam dimensions
    out_gif="beam_comparison.gif",
    fps=4,
)

Rendering frame 0/30
Rendering frame 1/30
Rendering frame 2/30
Rendering frame 3/30
Rendering frame 4/30
Rendering frame 5/30
Rendering frame 6/30
Rendering frame 7/30
Rendering frame 8/30
Rendering frame 9/30
Rendering frame 10/30
Rendering frame 11/30
Rendering frame 12/30
Rendering frame 13/30
Rendering frame 14/30
Rendering frame 15/30
Rendering frame 16/30
Rendering frame 17/30
Rendering frame 18/30
Rendering frame 19/30
Rendering frame 20/30
Rendering frame 21/30
Rendering frame 22/30
Rendering frame 23/30
Rendering frame 24/30
Rendering frame 25/30
Rendering frame 26/30
Rendering frame 27/30
Rendering frame 28/30
Rendering frame 29/30
Rendering frame 30/30
GIF saved => beam_comparison.gif
Done.


In [ ]:
# from IPython.display import Image

# Image(filename="beam_comparison.gif")